# FoSp + full MAT: train trên Kaggle
Bật **GPU + Internet**, sửa URL repo và đường dẫn dataset. Chạy từng cell, chỉ tiếp tục khi lệnh trước thành công. Xem `KAGGLE.md` trong repo để biết chi tiết.

In [ ]:
!git clone https://github.com/YOUR_NAME/YOUR_REPO.git /kaggle/working/FoSp
%cd /kaggle/working/FoSp
!python -m venv --system-site-packages /kaggle/working/fosp-venv
!/kaggle/working/fosp-venv/bin/python tools/setup_kaggle.py

In [ ]:
import os
os.environ['SMOKE_DATA_ROOT'] = '/kaggle/input/YOUR_DATASET/SmokeSeg'
os.environ['SMOKE_SPLIT_DIR'] = 'ImageSets/Segmentation'
# Nếu đã gắn pretrained dưới dạng Kaggle Dataset, bỏ comment và sửa:
# os.environ['MAT_CHECKPOINT'] = '/kaggle/input/YOUR_WEIGHTS/Places_512_FullData.pkl'
# os.environ['MIT_CHECKPOINT'] = '/kaggle/input/YOUR_WEIGHTS/mit_b3_20220624-13b1141c.pth'

In [ ]:
import subprocess
PY = '/kaggle/working/fosp-venv/bin/python'
if 'MAT_CHECKPOINT' not in os.environ:
    subprocess.run([PY, 'tools/download_mat.py'], check=True)
subprocess.run([PY, 'tools/check_kaggle_data.py', 'config/fosp_kaggle.py'], check=True)
subprocess.run([PY, '-m', 'pytest', 'tests', '-q'], check=True)

## Kiểm tra train → validation → lưu → resume
Chỉ dùng một ảnh train và một ảnh validation để kiểm tra luồng chạy. Không đo chất lượng. File checkpoint khá lớn; có thể xóa thư mục `fosp_check` sau khi PASS.

In [ ]:
subprocess.run([PY, 'tools/check_training.py', 'config/fosp_kaggle.py', '--work-dir', '/kaggle/working/fosp_check'], check=True)

## Train đầy đủ
40.000 iterations, batch 1, một GPU. MAT đủ hai stage và đóng băng; các phần còn lại được train. Lưu output phiên để giữ checkpoint.

In [ ]:
subprocess.run([PY, 'tools/train.py', 'config/fosp_kaggle.py', '--work-dir', '/kaggle/working/fosp_runs'], check=True)

## Resume ở phiên sau (tùy chọn)
Gắn output phiên cũ thành Input, thay đường dẫn bên dưới. Đặt `RESUME` thành đường dẫn thật rồi chạy cell này thay cell train mới ở trên.

In [ ]:
RESUME = None  # '/kaggle/input/YOUR_PREVIOUS_OUTPUT/fosp_runs/iter_1000.pth'
if RESUME:
    subprocess.run([PY, 'tools/train.py', 'config/fosp_kaggle.py', '--resume-from', RESUME, '--work-dir', '/kaggle/working/fosp_runs'], check=True)